# 🔗 Overlap graph

Builds and draws the read overlap graph used in the Lander-Waterman assembly model.

> **Self-contained.** No imports from other project notebooks.


In [ ]:
# !pip install networkx matplotlib
import matplotlib.pyplot as plt
import networkx as nx


#### `overlap`
*Check if the end of string 'a' overlaps with the beginning of string 'b'.*


In [ ]:
def overlap(a: str, b: str, min_length: int = 3) -> int:
    """Check if the end of string 'a' overlaps with the beginning of string 'b'.

    Args:
        a: First DNA sequence.
        b: Second DNA sequence.
        min_length: Minimum required overlap length.
    Returns:
        Length of the overlap, or 0 if none found.
    """
    start = 0
    while True:
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        if b.startswith(a[start:]):
            return len(a) - start
        start += 1


#### `build_overlap_graph`
*Construct an adjacency-list overlap graph of the given reads.*


In [ ]:
def build_overlap_graph(reads: list, min_length: int = 3) -> dict:
    """Construct an adjacency-list overlap graph of the given reads.

    Args:
        reads: List of DNA read sequences.
        min_length: Minimum overlap length to add an edge.
    Returns:
        dict mapping each read → list of (neighbour_read, overlap_length).
    """
    graph = {}
    for a in reads:
        graph[a] = []
        for b in reads:
            if a != b:
                olen = overlap(a, b, min_length)
                if olen > 0:
                    graph[a].append((b, olen))
    return graph


#### `draw_graph`
*Visualise the overlap graph with NetworkX / Matplotlib.*


In [ ]:
def draw_graph(graph: dict, figure_name: str) -> None:
    """Visualise the overlap graph with NetworkX / Matplotlib.

    Args:
        graph: Overlap graph adjacency list.
        figure_name: Used as the plot title and output filename (no extension).
    """
    G = nx.DiGraph()
    for src, neighbours in graph.items():
        for dest, weight in neighbours:
            G.add_edge(src, dest, weight=weight)

    # Skip rendering graphs that are too large (would hang)
    if len(G.nodes) > 250:
        print(
            f"Graph too large ({len(G.nodes)} nodes) – "
            f"skipping overlap graph for '{figure_name}'."
        )
        return

    pos = nx.spring_layout(G, seed=42)
    labels = nx.get_edge_attributes(G, "weight")

    fig, ax = plt.subplots(figsize=(10, 7))
    nx.draw(G, pos, ax=ax, with_labels=False, node_size=500)
    nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels=labels)
    ax.set_title(figure_name.replace("_", " ").title())
    plt.savefig(f"{figure_name}.png", dpi=150)
    plt.close(fig)
    print(f"Overlap graph saved to '{figure_name}.png'")


# =============================================================================
# MODULE: alignment_visualization  (pysam-free, pure Python)
# =============================================================================
